# Topic Validation: Cross-Phase Native-Speaker Review

Generates a review CSV for human validation of LLM-generated topic labels,
then applies validated decisions back to all per-phase topic CSVs.

**Workflow:**
1. Set `mode = "generate"` and run → produces `topic-review-sheet.csv`
2. Human reviewer opens CSV, fills `human_label`, `action`, `merge_to`, `notes`
3. Set `mode = "apply"` and run → updates all `topics_{phase}.csv` files

**Actions:**
- `confirm` — keep LLM label (or use `human_label` if provided)
- `rename` — replace with `human_label`
- `merge` — reassign comments to `merge_to` topic_id
- `reject` — move to noise cluster (`topic_id = -1`)

In [ ]:
# --- papermill parameters ---
crisis_name = "ateneo-drowning"  # default; overridden by papermill
mode = "generate"  # "generate" or "apply"
review_csv_path = f"../../02-analysis/output/{crisis_name}/topic-review-sheet.csv"

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

## Mode: Generate Review Sheet

Reads all `topics_{phase}.csv` files, extracts unique topics with
top terms and sample comments, and exports a review CSV for human validation.

In [ ]:
if mode == "generate":
    output_dir = f"../output/{crisis_name}"
    topic_files = sorted(glob.glob(os.path.join(output_dir, "topics_*.csv")))
    topic_files = [f for f in topic_files if "review-sheet" not in f and "transcripts" not in f]

    print(f"Found {len(topic_files)} phase CSVs")

    # Load all phases and tag with source phase
    frames = []
    for f in topic_files:
        phase = os.path.basename(f).replace("topics_", "").replace(".csv", "")
        df = pd.read_csv(f)
        df["source_phase"] = phase
        frames.append(df)
    all_topics = pd.concat(frames, ignore_index=True)

    # Filter out noise cluster
    labeled = all_topics[all_topics["topic_id"] != -1].copy()
    print(f"Total labeled comments: {len(labeled)}")
    print(f"Unique topics across all phases: {labeled.groupby(['topic_id', 'topic_label']).ngroups}")

    # Aggregate per unique topic across phases
    review_rows = []
    for (tid, label), group in labeled.groupby(["topic_id", "topic_label"]):
        # Get top terms from any row (identical per topic_id)
        top_terms = group["top_terms"].iloc[0] if "top_terms" in group.columns else ""

        # Sample up to 3 comments
        samples = group["text"].sample(min(3, len(group)), random_state=42).tolist()
        samples_str = " | ".join([s[:150] for s in samples])

        # Phases this topic appears in
        phases = ", ".join(sorted(group["source_phase"].unique()))

        review_rows.append({
            "topic_id": tid,
            "topic_label": label,
            "top_terms": top_terms,
            "sample_comments": samples_str,
            "phases": phases,
            "comment_count": len(group),
            "human_label": "",
            "action": "",  # confirm / rename / merge / reject
            "merge_to": "",  # target topic_id if action=merge
            "notes": ""  # free text, e.g. "split into X and Y"
        })

    review_df = pd.DataFrame(review_rows).sort_values("topic_id")
    os.makedirs(os.path.dirname(review_csv_path), exist_ok=True)
    review_df.to_csv(review_csv_path, index=False)
    print(f"\nExported {len(review_df)} topics to review sheet:")
    print(f"  {review_csv_path}")
    print(f"\nFill columns: human_label, action, merge_to, notes")
    display(review_df)

## Mode: Apply Validated Decisions

Reads the reviewed CSV and updates all per-phase `topics_{phase}.csv` files
with renames, merges, and rejections.

In [ ]:
if mode == "apply":
    # Load validated review sheet
    review_df = pd.read_csv(review_csv_path)
    print(f"Loaded review sheet with {len(review_df)} topics")

    # Build lookup dicts from validated decisions
    renames = {}
    merges = {}    # topic_id -> merge_to
    rejects = []   # list of topic_ids to reject

    for _, row in review_df.iterrows():
        tid = int(row["topic_id"])
        action = str(row["action"]).strip().lower()

        if action in ("confirm", "rename"):
            if pd.notna(row["human_label"]) and str(row["human_label"]).strip():
                renames[tid] = str(row["human_label"]).strip()
        elif action == "merge":
            if pd.notna(row["merge_to"]) and str(row["merge_to"]).strip():
                merges[tid] = int(row["merge_to"])
                # Also rename the merge target if human_label provided
                if pd.notna(row["human_label"]) and str(row["human_label"]).strip():
                    renames[int(row["merge_to"])] = str(row["human_label"]).strip()
        elif action == "reject":
            rejects.append(tid)

    print(f"Renames: {len(renames)}")
    print(f"Merges: {len(merges)}")
    print(f"Rejects: {len(rejects)}")

    # Apply to each phase CSV
    output_dir = f"../output/{crisis_name}"
    topic_files = sorted(glob.glob(os.path.join(output_dir, "topics_*.csv")))
    topic_files = [f for f in topic_files if "review-sheet" not in f and "transcripts" not in f]

    for f in topic_files:
        phase = os.path.basename(f).replace("topics_", "").replace(".csv", "")
        df = pd.read_csv(f)
        before = df["topic_id"].nunique()

        # Apply merges (reassign topic_id)
        for src, dst in merges.items():
            df.loc[df["topic_id"] == src, "topic_id"] = dst

        # Apply rejects (move to noise)
        for tid in rejects:
            df.loc[df["topic_id"] == tid, "topic_id"] = -1
            df.loc[df["topic_id"] == -1, "topic_label"] = np.nan

        # Apply renames
        for tid, label in renames.items():
            df.loc[df["topic_id"] == tid, "topic_label"] = label

        after = df["topic_id"].nunique()
        df.to_csv(f, index=False)
        print(f"  {phase}: {before} → {after} topics")

    print("\nDone. All phase CSVs updated.")